## Heart Attack Prediction

In [6]:
# load libraries
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from joblib import dump, load

In [7]:
# load the data
data = pd.read_csv('../data/heart.csv')

In [8]:
# check the target variable distribution
print(data.output.value_counts())

output
1    165
0    138
Name: count, dtype: int64


In [9]:
# check the first few rows of the data
data.head()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


### Data Preprocessing

In [10]:
### Data Preprocessing

# 1. Definiamo i tipi di colonne
numerical_cols = ['age', 'trtbps', 'chol', 'thalachh', 'oldpeak']
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exng', 'slp', 'caa', 'thall']

# 2. Creiamo il ColumnTransformer
# Questo oggetto gestirà lo scaling per le colonne numeriche
# e lascerà invariate ('passthrough') le categoriche.
transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', 'passthrough', categorical_cols)
    ],
    remainder='drop'
)

transformer

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['age', 'trtbps', 'chol', 'thalachh',
                                  'oldpeak']),
                                ('cat', 'passthrough',
                                 ['sex', 'cp', 'fbs', 'restecg', 'exng', 'slp',
                                  'caa', 'thall'])])

In [11]:
# Define the seed for reproducibility
seed = 42

# Define the steps for the pipeline
steps = [
    ('transformer', transformer),
    ('classifier', MLPClassifier(max_iter=1000, random_state=seed)) # Usiamo ANN
]

# Define the pipeline
ann_pipeline = Pipeline(steps=steps)

ann_pipeline

Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'trtbps', 'chol',
                                                   'thalachh', 'oldpeak']),
                                                 ('cat', 'passthrough',
                                                  ['sex', 'cp', 'fbs',
                                                   'restecg', 'exng', 'slp',
                                                   'caa', 'thall'])])),
                ('classifier', MLPClassifier(max_iter=1000, random_state=42))])

In [12]:
# Split the data into train and test sets
y = data['output']
X = data.drop(['output'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

In [13]:
# Define the hyperparameter grid for ANN (MLPClassifier)
# 'classifier__' è il prefisso per accedere ai parametri del modello nella pipeline
ann_param_grid = {
    'classifier__hidden_layer_sizes': [(12,), (20,), (10, 5)],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__alpha': [0.0001, 0.001],
    'classifier__learning_rate_init': [0.0001, 0.001]
}

### Modelling

In [14]:
# Definiamo i modelli da testare (solo la nostra pipeline ANN)
model_list = [('ann', ann_pipeline)]
model_params_grid = [ann_param_grid]

# Perform a GridSearch
best_models_gs = []
for (model_name, model), hp in zip(model_list, model_params_grid):
    grid = GridSearchCV(model, hp, cv=3, scoring='f1', n_jobs=-1, verbose=1)
    print(f"Running GridSearchCV for {model_name}...")
    grid.fit(X_train, y_train)
    print(f"Model: {model_name}")
    print(f"Best params: {grid.best_params_}, Best score: {grid.best_score_}")
    best_models_gs.append((model_name, grid.best_estimator_, grid.best_score_, grid.best_params_))

Running GridSearchCV for ann...
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Model: ann
Best params: {'classifier__activation': 'tanh', 'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (10, 5), 'classifier__learning_rate_init': 0.0001}, Best score: 0.8692007797270955


C:\Users\sasyc\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [15]:
# Get the best model
hof_model_gs = max(best_models_gs, key=lambda item:item[2])[1]

# Best model prediction and scores
y_pred_test = hof_model_gs.predict(X_test)
print(classification_report(y_test, y_pred_test))

f1_test = f1_score(y_test, y_pred_test)
print(f'F1 on test set: {f1_test}')

# save the transformer and the *entire* best pipeline
print("Saving files...")
dump(transformer, '../models/transformer.joblib')
dump(hof_model_gs, '../models/model.joblib')
print("Files saved to 'models/' folder.")

              precision    recall  f1-score   support

           0       0.83      0.86      0.85        29
           1       0.87      0.84      0.86        32

    accuracy                           0.85        61
   macro avg       0.85      0.85      0.85        61
weighted avg       0.85      0.85      0.85        61

F1 on test set: 0.8571428571428571
Saving files...
Files saved to 'models/' folder.


### Inference

In [16]:
# 1. Load *only* the full pipeline model
model = load('../models/model.joblib')

# 2. Define feature columns (in the correct order)
feat_cols = [
    'age', 'sex', 'cp', 'trtbps', 'chol', 'fbs', 'restecg',
    'thalachh', 'exng', 'oldpeak', 'slp', 'caa', 'thall'
]

# 3. Create a test DataFrame (usiamo una riga media come esempio)
test_row = X_train.mean().values
df = pd.DataFrame([test_row], columns = feat_cols)

# 4. Make a prediction *directly* with the pipeline
if (model.predict(df)==0):
    result = "Prediction: Low probability of heart attack."
else:
    result = "Prediction: High probability of heart attack."

print(result)

Prediction: Low probability of heart attack.
